# 실습5 — 반도체 이미지 denoising (Colab / A100)

배포된 `train_denoising_example.ipynb` 와 **같은 데이터 · 같은 노이즈 합성 · 같은 지표**를 쓰고
학습 쪽만 손본 파이프라인이다. 코드는 노트북에 박아 넣지 않고
[ds-practice](https://github.com/kithhooni-commits/ds-practice) 저장소의 `실습5/src/denoise/` 를 clone 해서 쓴다.
(로컬에서 돌린 것과 완전히 같은 코드라는 뜻이다.)

바꾼 것과 이유는 `실습5/README.md` 에 정리돼 있다. 요약하면:
median 채널 추가 · Charbonnier loss · cosine LR · 긴 학습 · rot90 증강 · 8× self-ensemble.

**실행 순서**: 런타임을 A100 으로 바꾸고 위에서부터 순서대로.

## 0. 런타임 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. 데이터 준비

Drive 에 **zip 그대로 올려 뒀어도 된다.** 아래 셀이 알아서 찾아 `/content/data` 로 푼다.

Colab 에서 Drive I/O 는 학습의 병목이라, 어차피 로컬 SSD 로 옮기는 게 맞다.
zip 을 로컬로 푸는 건 Drive 폴더를 통째로 복사하는 것보다 빠르다 (파일 7,568개 vs 압축파일 1개).

이미 풀린 폴더가 Drive 에 있으면 그것도 찾는다. 배포 zip 의
`test_noise_only/test_noise_only/` 같은 중첩 구조도 그대로 받는다.

In [ ]:
import os
import zipfile
from pathlib import Path

SEARCH_ROOT = Path("/content/drive/MyDrive")   # zip 또는 dataset 폴더가 있는 곳
WORK = Path("/content/data")                   # 여기로 푼다 (로컬 SSD)
DATA_ROOT = None                               # 직접 지정하려면 여기에 경로를 적는다

WANT = ("dataset", "log_denoising_example", "code_denoising")


def looks_like_dataset(p: Path) -> bool:
    return (p / "train").is_dir() and (p / "test_label").is_dir()


def find_dataset(root: Path) -> list[Path]:
    seen = [root] + [p for d in ("*", "*/*", "*/*/*") for p in root.glob(d) if p.is_dir()]
    return [p for p in seen if looks_like_dataset(p)]


WORK.mkdir(parents=True, exist_ok=True)

# 1) Drive 에 zip 으로 올려 뒀으면 로컬 SSD 로 푼다
zips = [z for d in ("*.zip", "*/*.zip") for z in SEARCH_ROOT.glob(d)]
for z in sorted(zips):
    tag = next((w for w in WANT if z.name.startswith(w)), None)
    if tag is None:
        continue
    if (WORK / tag).exists():
        print("이미 있음:", WORK / tag)
        continue
    print("푸는 중:", z.name, "->", WORK)
    with zipfile.ZipFile(z) as f:
        f.extractall(WORK)

# 2) dataset 폴더 위치 확정 — 로컬에 푼 것 우선, 없으면 Drive 에서 찾는다
if DATA_ROOT is None:
    cands = find_dataset(WORK) or find_dataset(SEARCH_ROOT)
    if not cands:
        print("MyDrive 안:", [x.name for x in list(SEARCH_ROOT.iterdir())[:30]])
        print("WORK 안  :", [x.name for x in list(WORK.iterdir())[:30]])
        raise SystemExit("dataset 을 못 찾았다. zip 이름이나 위치를 확인하고 DATA_ROOT 를 직접 적을 것")
    DATA_ROOT = cands[0]
    if len(cands) > 1:
        print("후보가 여럿이다. 첫 번째를 쓴다:", [str(c) for c in cands])

DATA_ROOT = Path(DATA_ROOT)
os.environ["DS_DATA"] = str(DATA_ROOT)
print()
print("DATA_ROOT =", DATA_ROOT)
for sub in ("train", "val", "test_label", "test_noise_only"):
    q = DATA_ROOT / sub
    n = len(list(q.glob("**/*.npy"))) if q.exists() else 0
    print(f"{'OK  ' if n else '없음'} {sub:<18} {n:>5} npy")

### dataset 이 아직 Drive 에 있으면 로컬로 복사

위 셀이 Drive 에 이미 풀려 있던 폴더를 골랐다면, 학습 내내 Drive 에서 파일을 읽게 된다.
그건 느리다. 아래 셀이 그 경우에만 로컬로 옮긴다. (이미 `/content` 면 아무것도 안 한다.)

In [ ]:
import shutil

if str(DATA_ROOT).startswith("/content/drive"):
    dest = WORK / "dataset"
    if not dest.exists():
        print("복사 중:", DATA_ROOT, "->", dest)
        shutil.copytree(DATA_ROOT, dest)
    DATA_ROOT = dest
    os.environ["DS_DATA"] = str(DATA_ROOT)

print("DATA_ROOT =", DATA_ROOT)
!du -sh "{DATA_ROOT}"

## 3. 코드 받기

In [ ]:
REPO = Path("/content/ds-practice")
if REPO.exists():
    !cd "{REPO}" && git pull --ff-only
else:
    !git clone --depth 1 https://github.com/kithhooni-commits/ds-practice.git "{REPO}"

SRC = REPO / "실습5" / "src" / "denoise"
RUNS = Path("/content/runs")
RUNS.mkdir(exist_ok=True)
!ls "{SRC}"

## 4. 지표 검증 — 먼저 여기부터

우리가 재는 PSNR/SSIM 이 채점 숫자와 같은지 확인한다. 배포된 예시 로그
(`log_denoising_example/00012_train/baseline_metrics.json`)의 mean/median/adaptive 성적을
우리 로더·우리 지표로 다시 재서 대조한다.

**`PSNR 최대 차이: 0.0000 dB — 일치` 가 나와야 다음으로 넘어간다.**
여기가 어긋나면 학습 결과도 믿을 수 없다.

(예시 로그를 Drive 에 안 올렸으면 대조 없이 우리 숫자만 나온다. 그래도 상관없다.)

In [ ]:
LOG_EXAMPLE = next(
    (p for p in (WORK / "log_denoising_example", SEARCH_ROOT / "log_denoising_example") if p.exists()),
    None,
)
if LOG_EXAMPLE is None:
    hits = [p for d in ("*/log_denoising_example", "*/*/log_denoising_example") for p in SEARCH_ROOT.glob(d)]
    LOG_EXAMPLE = hits[0] if hits else None

DEST = REPO / "실습5" / "data"
if LOG_EXAMPLE is not None and not (DEST / "log_denoising_example").exists():
    DEST.mkdir(parents=True, exist_ok=True)
    !cp -r "{LOG_EXAMPLE}" "{DEST}/"

print("예시 로그:", LOG_EXAMPLE or "없음 (대조 없이 우리 숫자만 나온다)")

!cd "{SRC}" && python check_baselines.py --data "{DATA_ROOT}"

## 5. 학습

A100 기준 설정이다. 로컬 6GB GPU 에서는 `--patch 128 --batch 16` 을 썼지만,
A100 이면 **크롭 없이 256² 전체**를 배치 32로 돌릴 수 있다 — test 와 완전히 같은 크기로
배우는 셈이라 크롭보다 유리하다.

| 인자 | A100 | 6GB 로컬 | 비고 |
|---|---|---|---|
| `--patch` | 256 | 128 | 256 이면 크롭이 사실상 없음 |
| `--batch` | 32 | 16 | |
| `--epochs` | 60 | 40 | 대략 40분 / 70분 |
| `--model` | `dncnn_plus` | | `dncnn` 으로 두면 배포 구조 그대로 |

`dncnn_plus` 는 median 3×3 결과를 두 번째 입력 채널로 넣은 것뿐이고 파라미터는 576개만 는다.
salt & pepper 가 유독 어려운 문제(입력 17.3 dB)를 겨냥한 설계다.

In [ ]:
!cd "{SRC}" && python train.py \
    --model dncnn_plus \
    --epochs 60 \
    --patch 256 \
    --batch 32 \
    --lr 3e-4 \
    --loss charbonnier \
    --workers 8 \
    --data "{DATA_ROOT}" \
    --out "{RUNS}" \
    --tag a100

### 같은 레시피로 구조만 원본 — 공정한 ablation

median 채널이 실제로 기여했는지 보려면 나머지 조건을 똑같이 두고 `dncnn` 을 한 번 더 돌린다.
시간이 급하면 건너뛰어도 제출에는 지장 없다.

In [ ]:
!cd "{SRC}" && python train.py \
    --model dncnn \
    --epochs 60 --patch 256 --batch 32 --lr 3e-4 --loss charbonnier --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag a100_ablation

## 6. 평가 — 제출값 산출

`test_noise_only` 를 입력으로 넣고 `test_label` 로 채점한다. conventional 비교군
(mean/median/adaptive)도 같은 데이터로 함께 재서 표로 낸다.

`--self-ensemble` 은 dihedral 8종으로 추론해 되돌려 평균내는 것이다. 학습 비용 0,
추론만 8배 느려지고 (100장이라 몇 초), 보통 0.2~0.4 dB 를 공짜로 준다.

In [ ]:
CKPT = sorted(RUNS.glob("*a100/checkpoints/checkpoint_best.ckpt"))[-1]
print("checkpoint:", CKPT)

# self-ensemble 없이 / 있게 각각 재서 그 차이도 발표 근거로 남긴다
!cd "{SRC}" && python evaluate.py "{CKPT}" --data "{DATA_ROOT}" --figures
print("=" * 70)
!cd "{SRC}" && python evaluate.py "{CKPT}" --data "{DATA_ROOT}" --self-ensemble --figures

### 노이즈 종류별 before/after

In [ ]:
from IPython.display import Image, display
for p in sorted(CKPT.parent.parent.glob("test_grid*.png")):
    print(p.name)
    display(Image(filename=str(p)))

## 7. 제출

마지막 셀이 찍은 `제출값 → PSNR_total ... SSIM_total ...` 두 숫자를
`denoising_challenge_score.xlsx` 에 소수 둘째자리로 적는다.

넘어야 할 기준선은 배포 예시 로그(DnCNN 10 epoch)의 **PSNR 30.51 / SSIM 0.8950** 이다.

In [ ]:
import shutil
OUT = Path("/content/drive/MyDrive/실습프로젝트/runs_mine")
OUT.mkdir(parents=True, exist_ok=True)
run = CKPT.parent.parent
shutil.copytree(run, OUT / run.name, dirs_exist_ok=True)
print("Drive 에 저장:", OUT / run.name)
!ls -la "{OUT}/{run.name}"